# g5.12xlarge / g6.12xlarge Spot Availability — '둘 다 0이 아닌' 경우 + 지속 구간

`2026-03-13` ~ `2026-03-18` 데이터에서 두 인스턴스의 spot availability를 비교/필터링합니다.

**데이터/정의 (다른 노트북 기준)**
- 데이터: 같은 폴더의 `2026-03-*.csv` (`gpu_spot_analysis_o.ipynb`, `scenario_ranking_*.ipynb` 와 동일)
- 컬럼: `InstanceType, AZ, DDDRequestTime, Total(=5), Success, Fail, RequestDateTime`
- `g5.12xlarge` / `g6.12xlarge` 는 둘 다 `us-west-2c` AZ만 존재하고 `Success` 는 0~5
- **spot availability = instance 수 = `Success`** (가용 spot instance 개수, 0~5). GPU 환산(`×4`) 아님.
- 타임스탬프 정렬: `scenario_ranking_*.ipynb` 와 동일하게 **5분 resample + ffill**

**참고:** `g5=1 & g6=1` 동시 발생은 데이터상 **0회**. 둘 다 같은 값(g5==g6>0)으로 나타나는 값은 `3`, `5` 뿐.

In [1]:
import pandas as pd
import glob, os

# ============== 설정 ==============
START_DATE = '2026-03-13'
END_DATE   = '2026-03-18'   # inclusive

TARGETS  = ['g5.12xlarge', 'g6.12xlarge']
RESAMPLE = '5min'          # scenario_ranking_*.ipynb 와 동일
# availability = Success (instance 수, 0~5). GPU 환산 안 함.
# ==================================

In [2]:
# ----- CSV 로드 (날짜 범위 필터) -----
start = pd.Timestamp(START_DATE)
end   = pd.Timestamp(END_DATE)

all_csvs = sorted(glob.glob('./2026-*.csv'))
csv_files = [f for f in all_csvs
             if start <= pd.Timestamp(os.path.basename(f).replace('.csv', '')) <= end]
print('Loading:', [os.path.basename(f) for f in csv_files])

df_raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
df_raw = df_raw[df_raw['InstanceType'].isin(TARGETS)].copy()
df_raw['RequestDateTime'] = pd.to_datetime(df_raw['RequestDateTime'])
df_raw = df_raw.sort_values('RequestDateTime').reset_index(drop=True)
print('rows per type:')
print(df_raw['InstanceType'].value_counts())

Loading: ['2026-03-13.csv', '2026-03-14.csv', '2026-03-15.csv', '2026-03-16.csv', '2026-03-17.csv', '2026-03-18.csv']
rows per type:
InstanceType
g5.12xlarge    855
g6.12xlarge    855
Name: count, dtype: int64


In [3]:
# ----- 인스턴스별 5분 resample + ffill, instance 수(Success) 결합 -----
cols = {}
for itype in TARGETS:
    sub = df_raw[df_raw['InstanceType'] == itype][['RequestDateTime', 'Success']].copy()
    sub = sub.set_index('RequestDateTime').sort_index()
    sub = sub[~sub.index.duplicated(keep='last')]
    sub.index = sub.index.floor(RESAMPLE)
    sub = sub[~sub.index.duplicated(keep='last')]
    sub = sub.resample(RESAMPLE).ffill()
    key = itype.split('.')[0]                 # 'g5' / 'g6'
    cols[f'{key}_avail'] = sub['Success']     # instance 수 (0~5)

avail = pd.DataFrame(cols).dropna().astype(int)
avail['sum_avail'] = avail['g5_avail'] + avail['g6_avail']
avail.index.name = 'time'
print(f'결합 후 timeline: {avail.index.min()} ~ {avail.index.max()}  ({len(avail)} points)')
avail.head()

결합 후 timeline: 2026-03-13 01:20:00 ~ 2026-03-18 23:50:00  (1711 points)


,g5_avail,g6_avail,sum_avail
time,,,
2026-03-13 01:20:00,0,5,5
2026-03-13 01:25:00,0,5,5
2026-03-13 01:30:00,0,5,5
2026-03-13 01:35:00,0,5,5
2026-03-13 01:40:00,0,5,5


## 필터: 둘 다 0이 아닌 경우 (`g5 > 0` and `g6 > 0`)

In [4]:
both_nonzero = (avail['g5_avail'] > 0) & (avail['g6_avail'] > 0)
f = avail[both_nonzero].copy()
print(f'둘 다 0이 아닌 경우: {len(f)} / {len(avail)} points')
with pd.option_context('display.max_rows', None):
    display(f.reset_index())

둘 다 0이 아닌 경우: 356 / 1711 points


,time,g5_avail,g6_avail,sum_avail
0,2026-03-15 06:00:00,2,5,7
1,2026-03-15 06:05:00,2,5,7
2,2026-03-15 06:10:00,1,5,6
3,2026-03-15 06:15:00,1,5,6
4,2026-03-15 07:30:00,4,5,9
5,2026-03-15 07:35:00,4,5,9
6,2026-03-15 08:00:00,2,5,7
7,2026-03-15 08:05:00,2,5,7
8,2026-03-15 08:10:00,3,5,8
9,2026-03-15 08:15:00,3,5,8


## (참고) (g5_avail, g6_avail) 조합별 개수

In [5]:
print(f.groupby(['g5_avail', 'g6_avail']).size().rename('count'))

g5_avail  g6_avail
1         3             4
          5            40
2         1             4
          4             2
          5            36
3         1             2
          3             4
          4             2
          5            34
4         2             2
          5            38
5         1            12
          2            10
          3             2
          4            10
          5           154
Name: count, dtype: int64


## 연속 지속 구간 (run-length) 분석

어떤 조건(mask)이 5분 그리드에서 **연속으로(중간에 끊김 없이)** 얼마나 오래 유지됐는지 계산합니다.
- `N개 슬롯` = `N × 5분` 커버 구간
- 표시 구간 `start ~ end` 는 각 슬롯 시작 시각 기준

In [6]:
STEP = pd.Timedelta(RESAMPLE)

def all_runs(mask):
    """연속(5분 간격) True 구간 목록 -> [(start, end, n_slots), ...]"""
    idx = mask.index
    runs = []
    s = prev = None
    n = 0
    for t, v in zip(idx, mask.values):
        if v:
            if s is None or (t - prev) != STEP:
                if s is not None:
                    runs.append((s, prev, n))
                s = t
                n = 1
            else:
                n += 1
            prev = t
        else:
            if s is not None:
                runs.append((s, prev, n))
                s = None
    if s is not None:
        runs.append((s, prev, n))
    return runs

def summarize(name, mask):
    runs = all_runs(mask)
    cnt = int(mask.sum())
    if not runs:
        print(f'[{name}] 발생 0회 -> 지속 구간 없음')
        return None
    s, e, n = max(runs, key=lambda r: r[2])
    print(f'[{name}] 총 {cnt} slots, 구간 {len(runs)}개, '
          f'최장 = {n} slots ({n*5}분)  {s} ~ {e} (+5분 ={e+STEP})')
    return (s, e, n)

In [7]:
g5a, g6a = avail['g5_avail'], avail['g6_avail']

# 1) 문자 그대로 둘 다 1 인 경우
summarize('g5==1 & g6==1', (g5a == 1) & (g6a == 1))

# 2) 1:1 동률 (g5 == g6, 둘 다 >0)
summarize('balanced g5==g6 (>0)', (g5a == g6a) & (g5a > 0))

# 3) 둘 다 0이 아닌 경우
summarize('both nonzero', (g5a > 0) & (g6a > 0))

[g5==1 & g6==1] 발생 0회 -> 지속 구간 없음
[balanced g5==g6 (>0)] 총 158 slots, 구간 34개, 최장 = 16 slots (80분)  2026-03-15 22:50:00 ~ 2026-03-16 00:05:00 (+5분 =2026-03-16 00:10:00)
[both nonzero] 총 356 slots, 구간 42개, 최장 = 42 slots (210분)  2026-03-15 11:00:00 ~ 2026-03-15 14:25:00 (+5분 =2026-03-15 14:30:00)


(Timestamp('2026-03-15 11:00:00'), Timestamp('2026-03-15 14:25:00'), 42)

### 1:1 동률(g5==g6>0) 구간 전체 (긴 순서)

In [8]:
runs = all_runs((g5a == g6a) & (g5a > 0))
runs_df = pd.DataFrame(
    [(s, e, n, n * 5, int(avail.loc[s, 'g5_avail'])) for s, e, n in runs],
    columns=['start', 'end', 'n_slots', 'minutes', 'value(g5=g6)'],
).sort_values('n_slots', ascending=False).reset_index(drop=True)
runs_df

,start,end,n_slots,minutes,value(g5=g6)
0,2026-03-15 22:50:00,2026-03-16 00:05:00,16,80,5
1,2026-03-17 08:30:00,2026-03-17 09:35:00,14,70,5
2,2026-03-15 09:20:00,2026-03-15 10:15:00,12,60,5
3,2026-03-16 00:40:00,2026-03-16 01:25:00,10,50,5
4,2026-03-15 08:20:00,2026-03-15 09:05:00,10,50,5
5,2026-03-15 12:00:00,2026-03-15 12:35:00,8,40,5
6,2026-03-15 20:40:00,2026-03-15 21:15:00,8,40,5
7,2026-03-18 15:20:00,2026-03-18 15:45:00,6,30,5
8,2026-03-15 19:20:00,2026-03-15 19:45:00,6,30,5
9,2026-03-15 21:30:00,2026-03-15 21:45:00,4,20,5
